[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/llama-certified/notebooks/day-07-gguf-conversion.ipynb#scrollTo=a1b2c3d4)

---
# Day 7 · GGUF Conversion and Custom Modelfiles
**certified-journeys / llama-certified** · Day 7 · Deployment Prep

> **Goal for today:** Convert a merged Llama checkpoint to GGUF format, quantize it to Q4_K_M, write a custom Modelfile, and load and test the model in Ollama via the REST API.


In [ ]:
%pip install -q requests huggingface_hub transformers


## Step 1 · llama.cpp Architecture and Conversion Pipeline

llama.cpp is the engine behind Ollama, LM Studio, and most local inference stacks.
It defines the GGUF format — a self-contained binary that packs weights, tokenizer,
and metadata into a single file.

**Conversion pipeline overview:**

| Stage | Input | Output | Tool |
|-------|-------|--------|------|
| 1 — Export | HuggingFace bf16 | F16 GGUF | `convert_hf_to_gguf.py` |
| 2 — Quantize | F16 GGUF | Q4_K_M GGUF | `llama-quantize` |
| 3 — Package | Q4_K_M GGUF + Modelfile | Ollama model | `ollama create` |

**Why two steps?** Converting directly from bf16 to Q4 skips calibration,
producing weight rounding errors. F16 is lossless; Q4_K_M quantizes with
importance-matrix calibration that preserves key-value outliers.

Reference: [llama.cpp development docs](https://github.com/ggerganov/llama.cpp/blob/master/docs/development/README.md)


In [ ]:
import subprocess
import os
import sys

# ── Simulate the llama.cpp clone step (read-only in Colab demo) ──
# In a real GPU machine you would run:
#   git clone https://github.com/ggerganov/llama.cpp
#   cd llama.cpp && make -j$(nproc)

# We inspect the conversion script's --help output to understand its interface.
# On a real machine: python llama.cpp/convert_hf_to_gguf.py --help

CONVERSION_COMMAND_TEMPLATE = '''
python llama.cpp/convert_hf_to_gguf.py \\
    /path/to/merged-checkpoint \\
    --outfile /path/to/output-f16.gguf \\
    --outtype f16
'''

QUANTIZE_COMMAND_TEMPLATE = '''
llama.cpp/llama-quantize \\
    /path/to/output-f16.gguf \\
    /path/to/output-q4_k_m.gguf \\
    Q4_K_M
'''

print('--- Conversion command ---')
print(CONVERSION_COMMAND_TEMPLATE)
print('--- Quantization command ---')
print(QUANTIZE_COMMAND_TEMPLATE)

# Quantization types available in llama.cpp
QUANT_TYPES = {
    'Q4_K_M': 'Most popular — 4-bit with mixed-precision K matrices. Best quality/size tradeoff.',
    'Q5_K_M': '5-bit mixed — higher quality, ~20% larger than Q4_K_M.',
    'Q8_0':   '8-bit — near-lossless, good for evaluation baselines.',
    'Q2_K':   '2-bit — aggressive compression, noticeable quality drop.',
    'F16':    'Full half-precision — reference format before quantization.',
}

print('\nAvailable quantization types:')
for qtype, desc in QUANT_TYPES.items():
    print(f'  {qtype:<10} {desc}')


### What just happened?
- **Two-step pipeline** is the correct approach: F16 first, quantize second.
- **`convert_hf_to_gguf.py`** handles tokenizer embedding, rope scaling, and metadata extraction automatically.
- **Q4_K_M** uses a mixed K-matrix strategy: attention layers stay at higher precision, FFN layers are 4-bit. This is why it beats naive Q4.
- **`llama-quantize`** reads the F16 GGUF and applies block-wise quantization — it never touches the original HuggingFace weights.


## Step 2 · Simulating the GGUF Conversion in Python

In Colab we cannot run `make` or spawn GPU processes, so we simulate the
conversion pipeline with a Python class that mirrors the real interface.
This lets you understand the data flow before running on a real machine.

The real `convert_hf_to_gguf.py` script:
- Loads `config.json` to determine architecture (LlamaForCausalLM, MistralForCausalLM, etc.)
- Reads `tokenizer.model` or `tokenizer.json` for vocabulary
- Iterates `model.safetensors.index.json` shards and converts each tensor
- Writes a single GGUF binary with a header, key-value metadata, and tensor data


In [ ]:
import json
import struct
import hashlib
from dataclasses import dataclass, field
from typing import Dict, List, Optional
from pathlib import Path

@dataclass
class GGUFMetadata:
    '''Mirrors the metadata section of a real GGUF file header.'''
    architecture: str = 'llama'
    model_name: str = ''
    context_length: int = 4096
    embedding_length: int = 2048
    block_count: int = 16
    feed_forward_length: int = 8192
    attention_head_count: int = 32
    attention_head_count_kv: int = 8   # GQA heads
    rope_freq_base: float = 500000.0   # Llama 3.x default
    vocab_size: int = 128256
    quantization_type: str = 'F16'
    file_size_bytes: int = 0
    extra: Dict = field(default_factory=dict)

    def to_dict(self) -> dict:
        return {
            'general.architecture': self.architecture,
            'general.name': self.model_name,
            f'{self.architecture}.context_length': self.context_length,
            f'{self.architecture}.embedding_length': self.embedding_length,
            f'{self.architecture}.block_count': self.block_count,
            f'{self.architecture}.feed_forward_length': self.feed_forward_length,
            f'{self.architecture}.attention.head_count': self.attention_head_count,
            f'{self.architecture}.attention.head_count_kv': self.attention_head_count_kv,
            f'{self.architecture}.rope.freq_base': self.rope_freq_base,
            'tokenizer.ggml.model': 'llama',
            'tokenizer.ggml.tokens': f'[vocab_size={self.vocab_size}]',
            'general.quantization_version': 2,
            **self.extra,
        }


class GGUFConverter:
    '''Simulates convert_hf_to_gguf.py behavior for learning purposes.'''

    SUPPORTED_ARCHITECTURES = ['LlamaForCausalLM', 'MistralForCausalLM', 'Phi3ForCausalLM']

    def __init__(self, checkpoint_dir: str, output_path: str, outtype: str = 'f16'):
        self.checkpoint_dir = Path(checkpoint_dir)
        self.output_path = Path(output_path)
        self.outtype = outtype.upper()

    def _detect_architecture(self, config: dict) -> str:
        '''Reads config.json model_type to select the right GGUF writer.'''
        arch = config.get('architectures', ['LlamaForCausalLM'])[0]
        if arch not in self.SUPPORTED_ARCHITECTURES:
            raise ValueError(f'Unsupported architecture: {arch}')
        return arch

    def _estimate_file_size(self, config: dict, quant: str) -> int:
        '''Approximate GGUF size based on parameter count and quantization.'''
        hidden = config.get('hidden_size', 2048)
        layers = config.get('num_hidden_layers', 16)
        intermediate = config.get('intermediate_size', 8192)
        vocab = config.get('vocab_size', 128256)

        # Rough parameter count (billions)
        params_b = (
            layers * (4 * hidden**2 + 3 * hidden * intermediate)  # attention + FFN
            + vocab * hidden  # embeddings
        ) / 1e9

        bits_per_param = {'F16': 16, 'Q8_0': 8, 'Q5_K_M': 5.5, 'Q4_K_M': 4.5, 'Q2_K': 2.5}
        bpp = bits_per_param.get(quant, 4.5)
        size_gb = params_b * bpp / 8
        return int(size_gb * 1024**3)

    def convert(self, config: dict) -> GGUFMetadata:
        '''Simulate the conversion and return populated metadata.'''
        arch = self._detect_architecture(config)
        size = self._estimate_file_size(config, self.outtype)
        meta = GGUFMetadata(
            model_name=config.get('_name_or_path', 'unknown'),
            context_length=config.get('max_position_embeddings', 4096),
            embedding_length=config.get('hidden_size', 2048),
            block_count=config.get('num_hidden_layers', 16),
            feed_forward_length=config.get('intermediate_size', 8192),
            attention_head_count=config.get('num_attention_heads', 32),
            attention_head_count_kv=config.get('num_key_value_heads', 8),
            vocab_size=config.get('vocab_size', 128256),
            quantization_type=self.outtype,
            file_size_bytes=size,
        )
        print(f'[convert] Architecture: {arch}')
        print(f'[convert] Output type:  {self.outtype}')
        print(f'[convert] Est. size:    {size / 1024**3:.2f} GB')
        print(f'[convert] Written to:   {self.output_path}')
        return meta


# Simulate a Llama-3.2-1B config.json
llama_1b_config = {
    'architectures': ['LlamaForCausalLM'],
    '_name_or_path': 'meta-llama/Llama-3.2-1B',
    'hidden_size': 2048,
    'num_hidden_layers': 16,
    'num_attention_heads': 32,
    'num_key_value_heads': 8,
    'intermediate_size': 8192,
    'max_position_embeddings': 131072,
    'vocab_size': 128256,
}

converter = GGUFConverter(
    checkpoint_dir='/content/merged-llama-1b',
    output_path='/content/llama-1b-f16.gguf',
    outtype='f16',
)
meta = converter.convert(llama_1b_config)

print('\n--- GGUF Metadata (key-value pairs) ---')
for k, v in list(meta.to_dict().items())[:8]:
    print(f'  {k}: {v}')


### What just happened?
- **Architecture detection** reads `config.json` — one class per model family (Llama, Mistral, Phi).
- **Metadata keys** like `llama.context_length` are GGUF-standard — Ollama reads these at load time.
- **GQA (Grouped Query Attention)** is captured in `attention_head_count_kv` — Llama-3.2 uses 8 KV heads vs 32 query heads.
- **Size estimate** shows why Q4_K_M is popular: ~0.7 GB for 1B vs ~2 GB for F16.


## Step 3 · Quantization: Q4_K_M Internals

GGUF quantization packs multiple weights into a single block. For Q4_K_M:

| Component | Bit width | Purpose |
|-----------|-----------|--------|
| Block scale | 6-bit | Per-block multiplier |
| Block min   | 6-bit | Per-block offset |
| Weights     | 4-bit | Quantized values |
| Superblock scale | FP16 | Scale-of-scales for stability |

The 'K' in Q4_K_M = K-quant method. The 'M' = medium — a balance between
Q4_K_S (small: fewer precision layers) and Q4_K_L (large: more FP16 layers).

Attention layers (`.attn_k`, `.attn_v`) stay at Q5 or Q6 in the K-quant scheme
because attention scores are more sensitive to quantization error.


In [ ]:
import numpy as np

def quantize_q4_block(weights: np.ndarray, block_size: int = 32) -> dict:
    '''
    Simulate Q4_K block quantization for one weight tensor slice.
    Real llama-quantize does this in C across the full GGUF file.
    '''
    assert len(weights) % block_size == 0, 'Weights must be a multiple of block_size'
    blocks = weights.reshape(-1, block_size)
    quant_blocks = []

    for block in blocks:
        # Compute per-block scale (max absolute value maps to ±7 in 4-bit signed)
        scale = np.max(np.abs(block)) / 7.0
        if scale == 0:
            scale = 1e-8  # avoid division by zero

        # Quantize: round to nearest integer in [-8, 7]
        q = np.clip(np.round(block / scale), -8, 7).astype(np.int8)
        # Dequantize: recover approximate floats
        dequant = q.astype(np.float32) * scale

        quant_blocks.append({
            'scale': float(scale),
            'quantized': q,
            'dequantized': dequant,
            'mse': float(np.mean((block - dequant) ** 2)),
        })

    return quant_blocks


# Simulate a small weight tensor from an attention layer
np.random.seed(42)
weights_f16 = np.random.randn(128).astype(np.float32) * 0.02  # typical weight magnitude

blocks = quantize_q4_block(weights_f16, block_size=32)

print(f'Input weights shape:  {weights_f16.shape}')
print(f'Number of Q4 blocks:  {len(blocks)}')
print()
for i, b in enumerate(blocks[:2]):  # show first 2 blocks
    print(f'Block {i}:')
    print(f'  Scale:     {b["scale"]:.6f}')
    print(f'  Quant[0:8]: {b["quantized"][:8]}')
    print(f'  Dequant[0:4]: {b["dequantized"][:4].round(5)}')
    print(f'  MSE:       {b["mse"]:.2e}')

# Memory comparison
f16_bytes = weights_f16.nbytes  # 4 bytes/weight (float32 in numpy)
q4_bytes = len(weights_f16) * 4 / 8  # 4 bits per weight + ~5% overhead
print(f'\nMemory: F32={f16_bytes}B → Q4≈{int(q4_bytes)}B ({f16_bytes/q4_bytes:.1f}x compression)')


### What just happened?
- **Block-wise quantization** finds the scale per 32-weight block — not globally.
- **MSE ~1e-5** is typical for Q4 on small weight values; larger models with higher weight variance see higher error.
- **~8x compression** from F32 to Q4 (real GGUF is ~8x from F16, since F16 is already 2 bytes).
- **K-quant improvement**: real Q4_K_M also stores a per-superblock scale in F16 to reduce accumulation error across 8 blocks — our simulation skips that.


## Step 4 · Writing a Modelfile

A Modelfile is Ollama's declarative model spec — similar to a Dockerfile.
It tells Ollama which GGUF to use and overrides inference parameters.

**Key Modelfile instructions:**

| Instruction | Effect |
|-------------|--------|
| `FROM` | Base GGUF file path or hub model |
| `SYSTEM` | System prompt injected before every conversation |
| `PARAMETER temperature` | Sampling temperature (0 = greedy, 2 = very creative) |
| `PARAMETER num_ctx` | Context window size in tokens |
| `PARAMETER top_p` | Nucleus sampling threshold |
| `PARAMETER top_k` | Top-k sampling limit |
| `PARAMETER stop` | Token sequences that end generation |
| `TEMPLATE` | Jinja2-style chat template (overrides built-in) |

The chat template in the Modelfile must match the format the model was trained on.
Llama 3.x uses the `<|begin_of_text|>...<|eot_id|>` format.


In [ ]:
from pathlib import Path

# ── Modelfile for a domain-specific fine-tuned Llama-3.2-1B ──

MODELFILE_CONTENT = '''FROM /content/llama-1b-q4_k_m.gguf

# System prompt — defines the model's persona and task scope
SYSTEM """
You are a helpful assistant specialized in answering questions about
machine learning infrastructure and local LLM deployment.
Be concise and technically precise. If you are unsure, say so.
"""

# Inference parameters
PARAMETER temperature 0.7
PARAMETER top_p 0.9
PARAMETER top_k 40
PARAMETER num_ctx 4096
PARAMETER repeat_penalty 1.1

# Stop tokens for Llama 3.x chat format
PARAMETER stop "<|eot_id|>"
PARAMETER stop "<|end_of_text|>"

# Chat template for Llama 3.x instruct format
TEMPLATE """{{ if .System }}<|start_header_id|>system<|end_header_id|>

{{ .System }}<|eot_id|>{{ end }}
{{ if .Prompt }}<|start_header_id|>user<|end_header_id|>

{{ .Prompt }}<|eot_id|>{{ end }}
<|start_header_id|>assistant<|end_header_id|>

{{ .Response }}<|eot_id|>"""
'''

# Write the Modelfile to disk
modelfile_path = Path('/content/Modelfile')
modelfile_path.write_text(MODELFILE_CONTENT)

print(f'Modelfile written to: {modelfile_path}')
print(f'Size: {modelfile_path.stat().st_size} bytes')
print()
print('To load this model in Ollama:')
print('  ollama create llama-1b-custom -f /content/Modelfile')
print()
print('--- Modelfile contents ---')
print(MODELFILE_CONTENT)


### What just happened?
- **`FROM` path** points to the GGUF on disk — Ollama copies it to its model store (`~/.ollama/models/`).
- **`TEMPLATE`** overrides Ollama's built-in chat template. Llama 3.x uses `<|start_header_id|>` tokens — getting this wrong causes gibberish output.
- **`PARAMETER stop`** can appear multiple times — Ollama treats each as an additional stop sequence.
- **`num_ctx 4096`** limits context to 4K even if the GGUF supports 128K — important for memory control.


## Step 5 · Loading and Testing via Ollama REST API

After running `ollama create`, the model is available via the REST API at
`http://localhost:11434`. Ollama exposes two endpoint families:

| Endpoint | Format | Use case |
|----------|--------|----------|
| `POST /api/generate` | Native Ollama format | Single-turn completions |
| `POST /api/chat` | Native Ollama chat format | Multi-turn conversations |
| `POST /v1/chat/completions` | OpenAI-compatible | Drop-in for OpenAI SDK |
| `GET /api/tags` | JSON list | List loaded models |
| `DELETE /api/delete` | JSON body | Remove a model |

We mock the Ollama server here so the notebook runs in Colab without a local GPU.


In [ ]:
import json
import time
from unittest.mock import patch, MagicMock
from datetime import datetime

# ── Mock Ollama REST client ──
# In production: import requests; response = requests.post('http://localhost:11434/api/...')

class MockOllamaClient:
    '''Simulates Ollama REST API responses for Colab demo.'''

    def __init__(self, base_url: str = 'http://localhost:11434'):
        self.base_url = base_url
        self._models = {'llama-1b-custom': {'size': 0.74e9, 'quantization': 'Q4_K_M'}}

    def list_models(self) -> dict:
        return {
            'models': [
                {
                    'name': name,
                    'size': info['size'],
                    'details': {'quantization_level': info['quantization']},
                    'modified_at': datetime.utcnow().isoformat() + 'Z',
                }
                for name, info in self._models.items()
            ]
        }

    def generate(self, model: str, prompt: str, stream: bool = False) -> dict:
        '''Simulate POST /api/generate'''
        if model not in self._models:
            return {'error': f'model "{model}" not found'}
        fake_response = (
            f'Q4_K_M quantization uses 4-bit weights with mixed-precision K matrices. '
            f'For the prompt "{prompt[:30]}...", the model would generate a contextual response. '
            f'This is a simulated Ollama response for Colab — run locally for real inference.'
        )
        return {
            'model': model,
            'created_at': datetime.utcnow().isoformat() + 'Z',
            'response': fake_response,
            'done': True,
            'context': [1, 2, 3, 4],  # token IDs (truncated)
            'total_duration': 1_234_567_890,  # nanoseconds
            'eval_count': 48,
            'eval_duration': 987_654_321,
        }

    def chat(self, model: str, messages: list) -> dict:
        '''Simulate POST /api/chat'''
        last_user_msg = next(
            (m['content'] for m in reversed(messages) if m['role'] == 'user'), ''
        )
        return {
            'model': model,
            'message': {
                'role': 'assistant',
                'content': f'[Mocked Ollama chat response] You asked: "{last_user_msg[:60]}"',
            },
            'done': True,
            'eval_count': 32,
            'eval_duration': 654_321_000,
        }

    def tokens_per_sec(self, response: dict) -> float:
        '''Compute decode tokens/sec from Ollama timing fields.'''
        return response['eval_count'] / (response['eval_duration'] / 1e9)


# ── Demo ──
client = MockOllamaClient()

# 1. List models (equivalent to: curl http://localhost:11434/api/tags)
print('=== Available models ===')
models = client.list_models()
for m in models['models']:
    size_gb = m['size'] / 1e9
    print(f"  {m['name']:<25} {size_gb:.2f} GB  quant={m['details']['quantization_level']}")

# 2. Single-turn generation
print('\n=== Generate ===')
resp = client.generate(
    model='llama-1b-custom',
    prompt='What is the difference between Q4_K_M and Q5_K_M quantization?'
)
print(f'Response: {resp["response"]}')
print(f'Tokens/sec: {client.tokens_per_sec(resp):.1f}')

# 3. Multi-turn chat
print('\n=== Chat ===')
messages = [
    {'role': 'user', 'content': 'How does Ollama load GGUF models into memory?'}
]
chat_resp = client.chat(model='llama-1b-custom', messages=messages)
print(f'Assistant: {chat_resp["message"]["content"]}')


### What just happened?
- **`/api/generate`** is single-turn — good for one-off prompts and batch evaluation.
- **`/api/chat`** maintains message history in the request body — Ollama does not store server-side session state.
- **`eval_count` / `eval_duration`** are the fields to compute tokens/sec — `total_duration` includes model load time and is misleading for steady-state benchmarks.
- **`context`** in the `/api/generate` response is the KV cache token IDs — pass it back in the next request for stateful continuation.


## Step 6 · ollama create and Model Registry

When you run `ollama create`, Ollama:
1. Parses the Modelfile and resolves `FROM`
2. Copies the GGUF to `~/.ollama/models/blobs/` (content-addressed by SHA256)
3. Writes a manifest to `~/.ollama/models/manifests/registry.ollama.ai/library/<name>/latest`
4. Stores Modelfile parameters in the manifest JSON

This means the original GGUF and Modelfile are only needed once — Ollama
manages its own blob store afterward.


In [ ]:
import hashlib
import json
from pathlib import Path

def simulate_ollama_create(modelfile_path: str, model_name: str, gguf_size_bytes: int = 740_000_000):
    '''
    Simulate what ollama create does behind the scenes.
    Real command: ollama create llama-1b-custom -f /content/Modelfile
    '''
    modelfile_content = Path(modelfile_path).read_text()

    # 1. Parse FROM line to get GGUF path
    from_line = next(l for l in modelfile_content.splitlines() if l.startswith('FROM'))
    gguf_path = from_line.split(None, 1)[1].strip()
    print(f'[ollama create] FROM: {gguf_path}')

    # 2. Simulate blob SHA256 (in reality Ollama hashes the actual GGUF bytes)
    fake_gguf_bytes = b'FAKE_GGUF_CONTENT_' + model_name.encode()
    blob_sha = 'sha256:' + hashlib.sha256(fake_gguf_bytes).hexdigest()
    print(f'[ollama create] Blob digest: {blob_sha[:32]}...')

    # 3. Build manifest (mirrors real Ollama manifest format)
    manifest = {
        'schemaVersion': 2,
        'mediaType': 'application/vnd.docker.distribution.manifest.v2+json',
        'config': {
            'mediaType': 'application/vnd.ollama.image.model',
            'digest': blob_sha,
            'size': gguf_size_bytes,
        },
        'layers': [
            {'mediaType': 'application/vnd.ollama.image.model', 'digest': blob_sha, 'size': gguf_size_bytes},
            {'mediaType': 'application/vnd.ollama.image.params', 'digest': 'sha256:params-placeholder', 'size': 512},
            {'mediaType': 'application/vnd.ollama.image.template', 'digest': 'sha256:tmpl-placeholder', 'size': 256},
        ]
    }

    manifest_path = Path(f'/tmp/ollama_manifests/{model_name}_latest.json')
    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    manifest_path.write_text(json.dumps(manifest, indent=2))

    print(f'[ollama create] Manifest written: {manifest_path}')
    print(f'[ollama create] Model available as: {model_name}')
    print(f'[ollama create] Run: ollama run {model_name}')
    return manifest


manifest = simulate_ollama_create(
    modelfile_path='/content/Modelfile',
    model_name='llama-1b-custom',
    gguf_size_bytes=740_000_000,  # ~0.74 GB for Llama-3.2-1B Q4_K_M
)

print(f'\nManifest layers: {len(manifest["layers"])}')
for layer in manifest['layers']:
    print(f'  {layer["mediaType"]:<55} {layer["size"]:>12,} bytes')


### What just happened?
- **Ollama uses OCI-style manifests** — the same format as Docker image layers.
- **Three layers** are typical: model weights (GGUF blob), params (from `PARAMETER` lines), and template (from `TEMPLATE` block).
- **Content-addressed storage** means if you create two models from the same GGUF, Ollama shares the blob — no duplication.
- **`ollama run` vs `ollama serve`**: `run` is interactive CLI, `serve` starts the API server daemonically.


In [ ]:
# Challenge: Write a Modelfile validator that checks for required fields
# and common mistakes before running ollama create.
#
# Your validator should:
#   1. Check that FROM is present and points to a .gguf file
#   2. Verify temperature is between 0 and 2 (if present)
#   3. Verify num_ctx is a power of 2 between 512 and 131072 (if present)
#   4. Warn if SYSTEM prompt is missing (not an error, but a best practice)
#   5. Warn if no stop tokens are defined for a Llama 3.x model
#   6. Return a dict: {'valid': bool, 'errors': [...], 'warnings': [...]}
#
# Test your validator against the MODELFILE_CONTENT defined earlier.

def validate_modelfile(content: str) -> dict:
    # Your solution here
    errors = []
    warnings = []

    # Scaffold: parse lines and check FROM
    lines = [l.strip() for l in content.splitlines() if l.strip() and not l.startswith('#')]
    # TODO: check FROM, PARAMETER temperature, PARAMETER num_ctx, SYSTEM, stop tokens

    return {'valid': len(errors) == 0, 'errors': errors, 'warnings': warnings}


# Test with the Modelfile we wrote earlier
result = validate_modelfile(MODELFILE_CONTENT)
print('Validation result:', json.dumps(result, indent=2))


---
## Day 7 key concepts recap

| Concept | What to remember |
|---|---|
| Two-step conversion | F16 GGUF first, then `llama-quantize` to Q4_K_M — never skip the F16 step |
| Q4_K_M | K-quant method with mixed precision; attention layers preserved at higher bit-depth |
| GGUF metadata | Architecture, context length, GQA heads, rope freq base — Ollama reads all of these at load time |
| Modelfile | Declarative spec: FROM + SYSTEM + PARAMETER + TEMPLATE; chat template must match training format |
| ollama create | OCI-style content-addressed blob store; three manifest layers: weights, params, template |
| REST API endpoints | `/api/generate` (single-turn), `/api/chat` (multi-turn), `/v1/chat/completions` (OpenAI-compat) |
| tokens/sec measurement | Use `eval_count / (eval_duration / 1e9)` — not `total_duration` which includes load time |

> **Tip:** Always convert to F16 GGUF first, then quantize to Q4_K_M as a second step. Converting directly to Q4 from a HuggingFace bf16 checkpoint skips the calibration pass and produces lower quality weights.

---
## What's next
**Day 8** → Serving with Ollama REST API and OpenAI-Compatible Endpoints — connect the OpenAI Python SDK to your local model, build a FastAPI proxy with rate limiting, and benchmark concurrent request throughput.

Mark Day 7 complete in your [tracker](../index.html).
